In [199]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [200]:
pip install alpaca-trade-api

In [201]:
pip install polygon-api-client

In [202]:
import alpaca_trade_api as tradeapi
from polygon import RESTClient
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
import numpy as np

In [203]:
ALPACA_API_KEY = "AK1RX6F8W6QX207XPLDF"
ALPACA_SECRET_KEY = "WaPoTTxkQBGzC51LajCdyw8Pl6svbINa9eDu9TMK"
ALPACA_BASE_URL = "https://api.alpaca.markets"

POLYGON_API_KEY = "Gkm8qM_seLdMVw3YRiSBODhwwmpPPpn4"

In [204]:
# Inicializa a API Alpaca com as credenciais lidas do arquivo
api = tradeapi.REST(
    ALPACA_API_KEY, ALPACA_SECRET_KEY, ALPACA_BASE_URL, api_version="v2"
)

In [205]:
clock = api.get_clock()

In [206]:
clock

Clock({   'is_open': False,
    'next_close': '2025-05-12T16:00:00-04:00',
    'next_open': '2025-05-12T09:30:00-04:00',
    'timestamp': '2025-05-09T19:31:42.103108532-04:00'})

In [207]:
client = RESTClient(POLYGON_API_KEY)

In [208]:
status = client.get_market_status()

In [209]:
status

MarketStatus(after_hours=True, currencies=MarketCurrencies(crypto='open', fx='closed'), early_hours=False, exchanges=MarketExchanges(nasdaq='extended-hours', nyse='extended-hours', otc='closed'), indicesGroups=MarketIndices(s_and_p='open', societe_generale='open', cgi='open', msci='open', ftse_russell='open', mstar='open', mstarc='open', cccy='open', nasdaq='closed', dow_jones='closed'), market='extended-hours', server_time='2025-05-09T19:31:42-04:00')

In [210]:
if status.market == "open":
    print("The market is open regular hours.")
elif status.early_hours:
    print("The market is open pre hours.")
elif status.after_hours:
    print("The market is open after hours.")
else:
    print("The market is closed.")

The market is open after hours.


In [211]:
delayed_safe_df = 5

now = datetime.now(timezone.utc)
start = (now - timedelta(days=delayed_safe_df)).isoformat()

print("Now (UTC):", now)
print("Start (UTC):", start)

Now (UTC): 2025-05-09 23:31:42.526632+00:00
Start (UTC): 2025-05-04T23:31:42.526632+00:00


In [212]:
symbol = "AAPL"
timeframe = "5Min"
data_source = "sip"
dataset_size = 36

In [213]:
# Fetch the historical data
bars = api.get_bars(symbol, timeframe, start, feed=data_source).df.tail(dataset_size)

In [214]:
bars

,close,high,low,trade_count,open,volume,vwap
timestamp,,,,,,,
2025-05-09 20:05:00+00:00,198.3000,198.5300,198.2500,252,198.5300,48044,198.455308
2025-05-09 20:10:00+00:00,198.3500,198.4977,198.2592,119,198.2800,37915,198.399256
2025-05-09 20:15:00+00:00,198.3500,198.4550,198.2927,166,198.4000,7761,198.364545
2025-05-09 20:20:00+00:00,198.3270,198.5000,198.3025,137,198.3500,165982,198.394211
2025-05-09 20:25:00+00:00,198.4600,198.4975,198.4050,23,198.4050,1002,198.447353
2025-05-09 20:30:00+00:00,198.3100,198.5300,198.3000,138,198.4800,6464,198.478315
2025-05-09 20:35:00+00:00,198.5300,198.5300,198.4700,78,198.4700,6204,198.514529
2025-05-09 20:40:00+00:00,198.5700,198.5700,198.5700,5,198.5700,372,198.570000
2025-05-09 20:45:00+00:00,198.4800,198.5400,198.3800,83,198.5300,4661,198.492890


In [215]:
data = bars[["vwap", "trade_count"]]

In [216]:
data

,vwap,trade_count
timestamp,,
2025-05-09 20:05:00+00:00,198.455308,252
2025-05-09 20:10:00+00:00,198.399256,119
2025-05-09 20:15:00+00:00,198.364545,166
2025-05-09 20:20:00+00:00,198.394211,137
2025-05-09 20:25:00+00:00,198.447353,23
2025-05-09 20:30:00+00:00,198.478315,138
2025-05-09 20:35:00+00:00,198.514529,78
2025-05-09 20:40:00+00:00,198.570000,5
2025-05-09 20:45:00+00:00,198.492890,83


In [217]:
scaler = joblib.load(
    "/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/minmax_ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min.pkl"
)

In [218]:
X_VWAP = data[["vwap"]].to_numpy()

X_VWAP_scaled = scaler.transform(X_VWAP)

In [219]:
X_VWAP_scaled

array([[0.25627434],
       [0.25614263],
       [0.25606107],
       [0.25613077],
       [0.25625564],
       [0.2563284 ],
       [0.25641349],
       [0.25654384],
       [0.25636265],
       [0.25647467],
       [0.25660785],
       [0.25675126],
       [0.25659828],
       [0.25652018],
       [0.25644658],
       [0.2563836 ],
       [0.25650086],
       [0.25630825],
       [0.25649684],
       [0.25652565],
       [0.2565625 ],
       [0.25657779],
       [0.25654924],
       [0.25647959],
       [0.25639336],
       [0.25651519],
       [0.25642989],
       [0.25647548],
       [0.25649556],
       [0.25649684],
       [0.25652817],
       [0.25652109],
       [0.25646638],
       [0.25649684],
       [0.25654384],
       [0.25650461]])

In [220]:
X_Trade_Count = data[["trade_count"]].to_numpy()

In [221]:
X_Trade_Count

array([[252],
       [119],
       [166],
       [137],
       [ 23],
       [138],
       [ 78],
       [  5],
       [ 83],
       [ 89],
       [ 83],
       [ 74],
       [129],
       [120],
       [ 53],
       [ 82],
       [ 42],
       [ 47],
       [ 15],
       [ 19],
       [ 30],
       [ 32],
       [ 52],
       [ 40],
       [ 31],
       [ 34],
       [ 58],
       [ 15],
       [ 22],
       [  6],
       [ 20],
       [ 42],
       [ 13],
       [  5],
       [ 22],
       [ 33]])

In [222]:
X_combined = np.concatenate([X_VWAP_scaled, X_Trade_Count], axis=1)

In [223]:
X_combined

array([[  0.25627434, 252.        ],
       [  0.25614263, 119.        ],
       [  0.25606107, 166.        ],
       [  0.25613077, 137.        ],
       [  0.25625564,  23.        ],
       [  0.2563284 , 138.        ],
       [  0.25641349,  78.        ],
       [  0.25654384,   5.        ],
       [  0.25636265,  83.        ],
       [  0.25647467,  89.        ],
       [  0.25660785,  83.        ],
       [  0.25675126,  74.        ],
       [  0.25659828, 129.        ],
       [  0.25652018, 120.        ],
       [  0.25644658,  53.        ],
       [  0.2563836 ,  82.        ],
       [  0.25650086,  42.        ],
       [  0.25630825,  47.        ],
       [  0.25649684,  15.        ],
       [  0.25652565,  19.        ],
       [  0.2565625 ,  30.        ],
       [  0.25657779,  32.        ],
       [  0.25654924,  52.        ],
       [  0.25647959,  40.        ],
       [  0.25639336,  31.        ],
       [  0.25651519,  34.        ],
       [  0.25642989,  58.        ],
 

In [224]:
X_Tensor = np.expand_dims(X_combined, axis=0)

In [225]:
X_Tensor

array([[[  0.25627434, 252.        ],
        [  0.25614263, 119.        ],
        [  0.25606107, 166.        ],
        [  0.25613077, 137.        ],
        [  0.25625564,  23.        ],
        [  0.2563284 , 138.        ],
        [  0.25641349,  78.        ],
        [  0.25654384,   5.        ],
        [  0.25636265,  83.        ],
        [  0.25647467,  89.        ],
        [  0.25660785,  83.        ],
        [  0.25675126,  74.        ],
        [  0.25659828, 129.        ],
        [  0.25652018, 120.        ],
        [  0.25644658,  53.        ],
        [  0.2563836 ,  82.        ],
        [  0.25650086,  42.        ],
        [  0.25630825,  47.        ],
        [  0.25649684,  15.        ],
        [  0.25652565,  19.        ],
        [  0.2565625 ,  30.        ],
        [  0.25657779,  32.        ],
        [  0.25654924,  52.        ],
        [  0.25647959,  40.        ],
        [  0.25639336,  31.        ],
        [  0.25651519,  34.        ],
        [  0

In [226]:
model = load_model(
    "/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min+fm=vwap+sm=trade_count+tm=+r=36+sort=False+rfm=False+rsm=False+rtm=False+d=+st=minmax+cts=[0]+Lb=True+e=500+es=True+cb=val_accuracy+p=100+bs=128+tl=0.41606152057647705+ta=0.8365758657455444.keras"
)

In [227]:
predictions = model.predict(X_Tensor)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 797ms/step


In [228]:
predictions

array([[0.9610386]], dtype=float32)

In [229]:
decisive_sensibility = 0.5

predicted_classes = (predictions >= decisive_sensibility).astype(int)

In [230]:
print(predictions)
print(predicted_classes)

[[0.9610386]]
[[1]]


In [231]:
def add_hours_skipping_dynamic(
    start_ts: pd.Timestamp,
    hours: float,
    dynamic_blackout: tuple[pd.Timestamp, pd.Timestamp] | None = None,
) -> pd.Timestamp:
    """
    Add `hours` to start_ts, skipping exactly one dynamic blackout:
      1) If start_ts is inside the blackout, warp to its end.
      2) If start_ts is before the blackout start, consume up to it,
         subtracting that from `hours`.
      3) Jump to blackout end, then add whatever remains.
    """
    current = start_ts
    remaining = hours

    if dynamic_blackout:
        db_start, db_end = dynamic_blackout

        # 1) If we begin inside the blackout, warp to its end
        if db_start <= current < db_end:
            current = db_end

        # 2) If we begin before the blackout, consume up to its start
        if current < db_start:
            # hours available before blackout
            avail = (db_start - current).total_seconds() / 3600.0
            if remaining <= avail:
                # we can finish entirely before blackout
                return current + timedelta(hours=remaining)
            # else consume up to the blackout start...
            remaining -= avail
            # ...and warp to the blackout end
            current = db_end

    # 3) No blackout (or we've just jumped past it): finish adding
    return current + timedelta(hours=remaining)


def get_prediction_timewindow_utc(
    data: pd.DataFrame,
    status,
    clock,
    hours_to_add: float = 3,
    pre_open_offset: float = 5.5,
) -> str:
    """
    Returns a UTC‐based prediction window string, applying only:
      • one dynamic blackout [next-midnight UTC, next_open–offset)
        if in after‐hours/closed.
    Prints the date only once when start & end share the same day.
    """

    def ensure_utc(ts):
        ts = pd.to_datetime(ts)
        return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

    # 1) Normalize inputs to UTC
    last_ts = ensure_utc(data.index[-1])

    next_open = ensure_utc(clock.next_open)

    # 2) Build dynamic blackout when in after-hours:
    #    from the NEXT UTC midnight after 'now'
    #    until (next_open – pre_open_offset)
    if status.after_hours or status.market == "closed":
        db_start = (last_ts + timedelta(days=1)).normalize()
        db_end = next_open - timedelta(hours=pre_open_offset)
        dynamic_blackout = (db_start, db_end)
    else:
        dynamic_blackout = None

    # 3) Compute end timestamp, skipping that blackout
    end_ts = add_hours_skipping_dynamic(last_ts, hours_to_add, dynamic_blackout)

    # 4) Format in UTC, date only once if same day
    date_fmt = "%b %-d"  # e.g. "May 9"
    time_fmt = "%-I:%M %p"  # e.g. " at 02:20 PM UTC"

    s_date, s_time = last_ts.strftime(date_fmt), last_ts.strftime(time_fmt)
    e_date, e_time = end_ts.strftime(date_fmt), end_ts.strftime(time_fmt)

    if s_date == e_date:
        return f"Prediction valid from: {s_date}, {s_time} until {e_time} (UTC)"
    else:
        return (
            f"Prediction valid from: {s_date}, {s_time} until {e_date}, {e_time} (UTC)"
        )

In [232]:
window_str = get_prediction_timewindow_utc(
    data, status, clock, hours_to_add=3, pre_open_offset=5.5
)
print(window_str)

Prediction valid from: May 9, 11:15 PM until May 12, 10:15 AM (UTC)
